# Importing the required libraries

In [72]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, month, udf, when, hour, dayofweek, month, date_format, desc, asc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.types import TimestampType, DoubleType, IntegerType, FloatType
from pyspark.sql.functions import col, radians, sin, cos, atan2, sqrt


# Initialising Spark
 Since the dataset we are working with has over a million records, we must allocate more memory to the spark initialisation so the worker doesn't crash

In [73]:
spark = SparkSession.builder \
    .appName("TaxiWeatherMerge") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()
print("Spark session successfully built")

Spark session successfully built


# Loading the data
In spark we must first specify the schema before loading the data to be sure it follows the same schema

In [74]:
taxi_schema = StructType([
    StructField("id", StringType(), True),
    StructField("vendor_id", IntegerType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("trip_duration", IntegerType(), True),
])
df_taxi = spark.read.schema(taxi_schema).csv("train.csv", header=True)
print("First few rows of the loaded dataset:")
df_taxi.show(5, truncate=True)

First few rows of the loaded dataset:
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-0

## Data conversion
- Creating a new column with the same values as pickup_ts but in timestamp format

In [75]:
df_taxi = df_taxi.withColumn("pickup_ts",to_timestamp(col("pickup_datetime"), "yyyy-MM-dd HH:mm:ss"))
print("Data types after converting pickup_datetime to timestamp:")
df_taxi.printSchema()


Data types after converting pickup_datetime to timestamp:
root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)



## Slicing dataset
- The dataset we work with is currently too big with over a million rows
- For convenience of data analytics or ML, we can sample the data either randomly or in a meaningful manner
- Therefore we slice the first 3 months of data

In [76]:
TARGET_MONTHS = [1, 2, 3]
df_jfm = df_taxi.filter(month(col("pickup_ts")).isin(TARGET_MONTHS))

## Missing value handling
- A missing value check is run using .isNull()
- Check is run on every column and returns where missing values are present
- Result returns no missing values

In [77]:
from pyspark.sql.functions import col, sum as spark_sum

total_rows = df_jfm.count()
Before_cleaning_count=total_rows
print("Total rows in dataset:", total_rows)

missingco = df_jfm.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_jfm.columns
])

missingco.show(truncate=False)


Total rows in dataset: 724196
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+---------+
|id |vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|pickup_ts|
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+---------+
|0  |0        |0              |0               |0              |0               |0              |0                |0               |0                 |0            |0        |
+---+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+---------+



## Cleaning passenger count data
- Check if the passenger count data is 0 and return the rows where it is
- Clean the data by filtering out the rows where passenger count is 0
- If passenger count is 0, it means that the ride is cancelled or it is an input error.
- Thereby making it an impossible value to be used in analytics except computing error itself.

In [78]:
zeropass = df_jfm.filter(col("passenger_count") == 0)
zeropass.show(truncate=False)
print("rows before cleaning",df_jfm.count())
df_jfm = df_jfm.filter(col("passenger_count") > 0)
print("rows after cleaning",df_jfm.count())

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+-------------------+
|id       |vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|pickup_longitude  |pickup_latitude   |dropoff_longitude |dropoff_latitude  |store_and_fwd_flag|trip_duration|pickup_ts          |
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+-------------------+
|id3645383|2        |2016-01-01 05:01:32|2016-01-01 05:01:36|0              |-73.99313354492188|40.75747299194336 |-73.99329376220703|40.757537841796875|N                 |4            |2016-01-01 05:01:32|
|id2840829|2        |2016-02-21 01:33:52|2016-02-21 01:36:27|0              |-73.94624328613281|40.77290344238281 |-73.94676971435547|40.77484130859375 |N                 |

## Cleaning up Trip duration outliers
- We have defined 60 seconds as the min extreme end and 4 hours as the maximum extreme end.
- We have filtered out the trips in those extreme ends
- We are filtering trips below 60 seconds as its not a usual value that can be used in analytics.
- We are filtering 4 hours away as it the extreme end that can skew results

In [79]:

mintripdur = 60   
maxtripdur = 14400 

print("rows before cleaning",df_jfm.count())

# Keep only rows where trip_duration is between 1 minute and 4 hours
df_jfm = df_jfm.filter(
    (col("trip_duration") >= mintripdur) &
    (col("trip_duration") <= maxtripdur)
)

# Calculate and report the changes
print("rows after cleaning",df_jfm.count())

rows before cleaning 724160
rows after cleaning 719033


## Cleaning passenger count for excessive amounts of passengers
- NYC yellow cabs generally do not have space for more than 5 people
- Any data showing so is an exceptional situation or abnormal
- Thereby we remove such data

In [80]:
print("rows before cleaning",df_jfm.count())
df_jfm = df_jfm.filter(col("passenger_count") <= 6)
print("rows after cleaning",df_jfm.count())


rows before cleaning 719033
rows after cleaning 719032


## Datatype conversion
- The datatypes for Pickup_datetime() and dropoff_datetime are converted to Timestamp type.
- The datatypes for the cordinates are converted to Double for more accuracy
- The datatype for trip_duration is converted to Integer since the trip duration is in seconds

In [81]:
df_jfm = df_jfm.withColumn("pickup_datetime", col("pickup_datetime").cast(TimestampType()))
df_jfm = df_jfm.withColumn("dropoff_datetime", col("dropoff_datetime").cast(TimestampType()))
coordinatcs = [
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude"
]
for column in coordinatcs:
    df_jfm = df_jfm.withColumn(column, col(column).cast(DoubleType()))
df_jfm = df_jfm.withColumn("trip_duration", col("trip_duration").cast(IntegerType()))
print("After conversion",df_jfm.printSchema())

root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)

After conversion None


## Cleaning values outside of NYC
- Since we are making a pipeline for NYC cab analytics, we do not require trips that start or end outside of NYC.
- That is, a NYC trip should start and stop in NYC
- Using the NYC bounding boxes we evaluate and remove trips that fall outside the boundaries

In [82]:
print("rows before cleaning",df_jfm.count())
minlong = -74.05
maxlong = -73.75
minlatt = 40.60
maxlatt = 40.90
geofilter = (
    (col("pickup_longitude") >= minlong) &
    (col("pickup_longitude") <= maxlong) &

    (col("pickup_latitude") >= minlatt) &
    (col("pickup_latitude") <= maxlatt) &

    (col("dropoff_longitude") >= minlong) &
    (col("dropoff_longitude") <= maxlong) &

    (col("dropoff_latitude") >= minlatt) &
    (col("dropoff_latitude") <= maxlatt)
)
df_jfm = df_jfm.filter(geofilter)
print("rows after cleaning",df_jfm.count())

rows before cleaning 719032
rows after cleaning 715857


## Haversine formula column
- We'll employ the Haversine formula to get a distance value
- we must first create temporary columns in radians using the coordinate columns
- Then we calculate the a value and c value in the formula
- We then calculate the Haversine formula in kms and create a seperate column to store them
- We then drop the temporary columns we created

In [83]:
R = 6371

df_jfm = df_jfm.withColumn("lat1", radians(col("pickup_latitude"))) \
               .withColumn("lon1", radians(col("pickup_longitude"))) \
               .withColumn("lat2", radians(col("dropoff_latitude"))) \
               .withColumn("lon2", radians(col("dropoff_longitude")))

df_jfm = df_jfm.withColumn("a",
    sin((col("lat2") - col("lat1")) / 2)**2 +
    cos(col("lat1")) * cos(col("lat2")) *
    sin((col("lon2") - col("lon1")) / 2)**2
)

df_jfm = df_jfm.withColumn("c", 2 * atan2(sqrt(col("a")), sqrt(1 - col("a"))))

df_jfm = df_jfm.withColumn("haversine_distance", col("c") * R)

df_jfm = df_jfm.drop("lat1", "lon1", "lat2", "lon2", "a", "c")

print("\nFirst 5 rows with Optimized Haversine Distance:")
df_jfm.select(
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "haversine_distance"
).show(5)



First 5 rows with Optimized Haversine Distance:
+------------------+------------------+------------------+------------------+------------------+
|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|haversine_distance|
+------------------+------------------+------------------+------------------+------------------+
| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|1.4985207796469109|
| -73.9790267944336|40.763938903808594|-74.00533294677734|40.710086822509766| 6.385098495252495|
|-73.97305297851562|40.793209075927734| -73.9729232788086| 40.78252029418945|1.1885884593338851|
|-73.98285675048828| 40.74219512939453|-73.99208068847656|40.749183654785156|1.0989424593055537|
|-73.98104858398438| 40.74433898925781| -73.9729995727539| 40.78998947143555| 5.121161562140773|
+------------------+------------------+------------------+------------------+------------------+
only showing top 5 rows


## Turn the store and forward flag column into binary columns
- This is for better machine interpretability for ML algorithms

In [84]:
df_jfm = df_jfm.withColumn(
    "store_and_fwd_flag",
    when(col("store_and_fwd_flag") == "Y", 1).otherwise(0).cast("int")
)
df_jfm.select("store_and_fwd_flag").distinct().show()

+------------------+
|store_and_fwd_flag|
+------------------+
|                 0|
|                 1|
+------------------+



## Extract hour, day and month
- Sprak provides inbuilt function for extracting these values

In [85]:
df_jfm = df_jfm.withColumn(
    "pickup_datetime_ts",
    to_timestamp(col("pickup_datetime"), "yyyy-MM-dd HH:mm:ss")
)
df_jfm = df_jfm.withColumn("hour_of_day", hour(col("pickup_datetime_ts")))

df_jfm = df_jfm.withColumn("day_of_week", dayofweek(col("pickup_datetime_ts")))

df_jfm = df_jfm.withColumn("month", month(col("pickup_datetime_ts")))

df_jfm = df_jfm.drop("pickup_datetime_ts", "pickup_datetime")
df_jfm.printSchema()
df_jfm.select(
    "hour_of_day",
    "day_of_week",
    "month", "pickup_ts"
).show(5)


root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = false)
 |-- trip_duration: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- haversine_distance: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)

+-----------+-----------+-----+-------------------+
|hour_of_day|day_of_week|month|          pickup_ts|
+-----------+-----------+-----+-------------------+
|         17|          2|    3|2016-03-14 17:24:55|
|         11|          3|    1|2016-01-19 11:35:24|
|         13|          7|    3|2016-03-26 13:30:5

# Sorting the dataset
- The dataset is not in order of timestamp.
- For ease , we can sort them by using the orderBy function and asc

In [86]:
df_jfm = df_jfm.orderBy(col("pickup_ts").asc())
df_jfm.select("pickup_ts", "trip_duration", "haversine_distance").show(5, truncate=False)

+-------------------+-------------+------------------+
|pickup_ts          |trip_duration|haversine_distance|
+-------------------+-------------+------------------+
|2016-01-01 00:00:17|849          |12.75662024810217 |
|2016-01-01 00:00:53|1294         |4.010131326363453 |
|2016-01-01 00:01:01|408          |2.170871624812562 |
|2016-01-01 00:01:14|280          |0.770146554454693 |
|2016-01-01 00:01:20|736          |2.4745753367206036|
+-------------------+-------------+------------------+
only showing top 5 rows


## Calculating the speed 
- Lets calculate speed and place it in a column
- Speed is distance by time. We must multiply by 3600 to standardise it to kilometers per hour

In [87]:
df_jfm = df_jfm.withColumn(
    "avg_speed_kph",
    (col("haversine_distance") / col("trip_duration")) * 3600
)
df_jfm.select("pickup_ts", "trip_duration", "haversine_distance", "avg_speed_kph").show(5, truncate=False)

+-------------------+-------------+------------------+------------------+
|pickup_ts          |trip_duration|haversine_distance|avg_speed_kph     |
+-------------------+-------------+------------------+------------------+
|2016-01-01 00:00:17|849          |12.75662024810217 |54.09167596368411 |
|2016-01-01 00:00:53|1294         |4.010131326363453 |11.156470459743764|
|2016-01-01 00:01:01|408          |2.170871624812562 |19.15474963069908 |
|2016-01-01 00:01:14|280          |0.770146554454693 |9.90188427156034  |
|2016-01-01 00:01:20|736          |2.4745753367206036|12.103901103524692|
+-------------------+-------------+------------------+------------------+
only showing top 5 rows


## Displaying fastest and slowest trips
- Lets display the fastest and slowest trips to look for any outliers
- We do this by desc and asc

In [88]:
df_jfm.select("haversine_distance", "trip_duration", "avg_speed_kph").orderBy(desc("avg_speed_kph")).show(10, truncate=False)
df_jfm.select("haversine_distance", "trip_duration", "avg_speed_kph").orderBy(asc("avg_speed_kph")).show(10, truncate=False)

+------------------+-------------+------------------+
|haversine_distance|trip_duration|avg_speed_kph     |
+------------------+-------------+------------------+
|19.61995884113818 |121          |583.7343126289045 |
|21.715108237793505|184          |424.8608133481338 |
|18.31832429812108 |207          |318.5795530108014 |
|20.144298574511385|268          |270.59505547851114|
|13.267623296858442|187          |255.41948592882562|
|11.438519312353938|248          |166.04302227610555|
|18.756283164279854|420          |160.76814140811302|
|14.233009862152416|355          |144.33474789788366|
|18.90920406627235 |637          |106.86520351425503|
|13.258514560269106|450          |106.06811648215285|
+------------------+-------------+------------------+
only showing top 10 rows
+------------------+-------------+-------------+
|haversine_distance|trip_duration|avg_speed_kph|
+------------------+-------------+-------------+
|0.0               |1336         |0.0          |
|0.0               |796

# Observation
- we can see some very unrealistic values in the displayed results.
- Distance being 0 is a classic case of pickup and dropoff in the same coordinates
- This is most likely just an error in reading or input. We can remove it
- As for the average speed above 120kmph those are exceptions or highly unrealistic. Therefore we can remove them

## Remove the outliers

In [89]:
zerodist = df_jfm.filter(col("haversine_distance") == 0.0)
zerodist.select(
    "pickup_longitude", 
    "pickup_latitude", 
    "dropoff_longitude", 
    "dropoff_latitude", 
    "trip_duration"
).limit(10).show(truncate=False)
zerodist.select("trip_duration").describe().show()

+------------------+------------------+------------------+------------------+-------------+
|pickup_longitude  |pickup_latitude   |dropoff_longitude |dropoff_latitude  |trip_duration|
+------------------+------------------+------------------+------------------+-------------+
|-73.9461669921875 |40.75090789794922 |-73.9461669921875 |40.75090789794922 |1674         |
|-73.91694641113281|40.83726119995117 |-73.91694641113281|40.83726119995117 |2074         |
|-73.98982238769531|40.73530197143555 |-73.98982238769531|40.73530197143555 |945          |
|-73.79338836669922|40.6568603515625  |-73.79338836669922|40.6568603515625  |884          |
|-73.93392944335938|40.85329055786133 |-73.93392944335938|40.85329055786133 |298          |
|-73.9720687866211 |40.762840270996094|-73.9720687866211 |40.762840270996094|865          |
|-73.96236419677734|40.80074691772461 |-73.96236419677734|40.80074691772461 |1358         |
|-73.95466613769531|40.82100296020508 |-73.95466613769531|40.82100296020508 |641

In [90]:
print("rows before cleaning",df_jfm.count())
mindist = 0.01  # Minimum distance to ensure we exclude zero-distance trips
maxdist = 120.0   # Maximum plausible Haversine speed in an urban setting
df_jfm = df_jfm.filter(
    (col("haversine_distance") >= mindist) & 
    (col("avg_speed_kph") <= maxdist)
)
print("rows after cleaning",df_jfm.count())


rows before cleaning 715857
rows after cleaning 713541


## Adding label columns
- For now the columns that point to weeks and months are displayed by numbers
- For ease of human readability let us convert them to their respective string labels

In [91]:
df_jfm = df_jfm.withColumn(
    "day_of_week_label",
    when(col("day_of_week") == 1, "Sunday")
    .when(col("day_of_week") == 2, "Monday")
    .when(col("day_of_week") == 3, "Tuesday")
    .when(col("day_of_week") == 4, "Wednesday")
    .when(col("day_of_week") == 5, "Thursday")
    .when(col("day_of_week") == 6, "Friday")
    .when(col("day_of_week") == 7, "Saturday")
    .otherwise("Error_Value")
)

df_jfm = df_jfm.withColumn(
    "month_label",
    when(col("month") == 1, "January")
    .when(col("month") == 2, "February")
    .when(col("month") == 3, "March")
    .otherwise("Error_Value_OOT")
)
df_jfm.filter(col("day_of_week") == 1).select(
    "pickup_ts","day_of_week_label", 
    "day_of_week", "month_label",
    "hour_of_day", 
    "month"
).show(5, truncate=False)


+-------------------+-----------------+-----------+-----------+-----------+-----+
|pickup_ts          |day_of_week_label|day_of_week|month_label|hour_of_day|month|
+-------------------+-----------------+-----------+-----------+-----------+-----+
|2016-01-03 00:00:12|Sunday           |1          |January    |0          |1    |
|2016-01-03 00:00:14|Sunday           |1          |January    |0          |1    |
|2016-01-03 00:00:23|Sunday           |1          |January    |0          |1    |
|2016-01-03 00:00:32|Sunday           |1          |January    |0          |1    |
|2016-01-03 00:00:39|Sunday           |1          |January    |0          |1    |
+-------------------+-----------------+-----------+-----------+-----------+-----+
only showing top 5 rows


## Saving the final file

In [92]:
(
    df_jfm
    .coalesce(1)
    .write.csv("enriched_taxi_data_sorted_by_time.csv", header=True, mode="overwrite")
)

In [93]:
df_jfm.printSchema()


root
 |-- id: string (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: integer (nullable = false)
 |-- trip_duration: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- haversine_distance: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- avg_speed_kph: double (nullable = true)
 |-- day_of_week_label: string (nullable = false)
 |-- month_label: string (nullable = false)

